In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\MLOps\02_End_to_End_Projects\Movie-Sentiment-Classification\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\BAPUN
[nltk_data]     SUNA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\BAPUN
[nltk_data]     SUNA\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
df = pd.read_csv("IMDB.csv")
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
814,"The Shining, you know what's weird about this ...",positive
970,I saw this back in '94 when it was finally rel...,negative
761,I was very lucky to see this film as part of t...,positive
839,I and my brother are very big Asian movie fans...,negative
19,I tend to get furious when hearing about Lucio...,positive


In [4]:
# Data preprocessing

# Define the text preprocessing functions
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace(';', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [5]:
df = normalize_text(df)
df.head()

,review,sentiment
814,shining know what s weird movie movie everyone...,positive
970,saw back finally released apparently orion pic...,negative
761,lucky see film part melbourne international fi...,positive
839,brother big asian movie fan finding movie hidd...,negative
19,tend get furious hearing lucio fulci s reputat...,positive


In [6]:
df['sentiment'].value_counts()

sentiment
negative    270
positive    230
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
814,shining know what s weird movie movie everyone...,1
970,saw back finally released apparently orion pic...,0
761,lucky see film part melbourne international fi...,1
839,brother big asian movie fan finding movie hidd...,0
19,tend get furious hearing lucio fulci s reputat...,1


In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [10]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/BapunSuna/Movie-Sentiment-Classification.mlflow')
dagshub.init(repo_owner='BapunSuna',
             repo_name='Movie-Sentiment-Classification',
             mlflow=True)
mlflow.set_experiment("LogisticRegression Baseline")

Initialized MLflow to track repo "BapunSuna/Movie-Sentiment-Classification"

Repository BapunSuna/Movie-Sentiment-Classification initialized!

<Experiment: artifact_location='mlflow-artifacts:/31f4ba32bc1348cfbc6932277c3a8140', creation_time=1790016750432, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1790016750432, lifecycle_stage='active', name='LogisticRegression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [15]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log Exection time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occured: {e}", exc_info=True)

2026-09-22 00:37:05,630 - INFO - Starting MLflow run...
2026-09-22 00:37:07,036 - INFO - Logging preprocessing parameters...
2026-09-22 00:37:08,139 - INFO - Initializing Logistic Regression model...
2026-09-22 00:37:08,140 - INFO - Fitting the model...
2026-09-22 00:37:08,163 - INFO - Model training complete.
2026-09-22 00:37:08,164 - INFO - Logging model parameters...
2026-09-22 00:37:08,535 - INFO - Making predictions...
2026-09-22 00:37:08,536 - INFO - Calculating evaluation metrics...
2026-09-22 00:37:08,545 - INFO - Logging evaluation metrics...
2026-09-22 00:37:10,115 - INFO - Saving and logging the model...
2026/09/22 00:37:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/22 00:37:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026-09-22 00:37:23,519 - INFO - Model training and logging completed in 16.48 seconds.
2026-

🏃 View run delicate-donkey-316 at: https://dagshub.com/BapunSuna/Movie-Sentiment-Classification.mlflow/#/experiments/0/runs/4014d943bfae4a88b84fa4f64cbca9f6
🧪 View experiment at: https://dagshub.com/BapunSuna/Movie-Sentiment-Classification.mlflow/#/experiments/0
